# Lesson 09: Reading, and critiquing, our own results
⏱ **60 min** · files you will read: `reports/RESULTS.md`, `results/smoke/stats/*.parquet`,
`reports/dashboard/index.html`

**You'll be able to…**
1. trace any number in the report back to the stored table it came from;
2. tell a *surviving* finding from one that only looked significant before correction;
3. tell an *absent* result from a *null* result;
4. run a critique checklist on any forecasting study, starting with this one.

**You need:** Lessons 06–08.

In [ ]:
import pandas as pd

from tsfm_rc.paths import REPORTS_DIR, RESULTS_DIR

stats = RESULTS_DIR / "smoke" / "stats"
dm_all = pd.read_parquet(stats / "dm_all.parquet")
dm_primary = pd.read_parquet(stats / "dm_primary.parquet")
mcs = pd.read_parquet(stats / "mcs.parquet")
print((REPORTS_DIR / "smoke" / "RESULTS.md").read_text()[:900])

The banner matters most: these are **synthetic** fixtures. Nothing below is about markets.

## 1. Trace a number
Section 4 of the report lists `garch` vs `har` for rv. Find the same row in the table.

In [ ]:
row = dm_all.query("period == 'full' and window == 'expanding' and target == 'rv' and model == 'garch' and horizon == 5")
print(row[["rel_loss", "rel_loss_lo", "rel_loss_hi", "dm_stat", "p_value", "p_holm", "T"]].round(3).to_string(index=False))

Same numbers as the report, because the report is *rendered* from this file (a test checks
that re-rendering gives an identical file).

## 2. Raw vs corrected
🤔 **Predict before you run:** of the tests with raw p < 0.05, what fraction survive Holm?

In [ ]:
fam = dm_all.query("period == 'full' and window == 'expanding'")
raw = fam["p_value"] < 0.05
print(f"{raw.sum()} raw-significant, {fam['reject_holm'].sum()} survive Holm")
print(fam.loc[raw & ~fam["reject_holm"], ["target", "horizon", "model", "p_value", "p_holm"]].round(3).to_string(index=False))

Those rows are what a less careful study would have published.

## 3. Absent is not null
🤔 **Predict:** what does the primary table say about the TSFMs in this run?

In [ ]:
print(dm_primary["status"].value_counts())

"UNAVAILABLE" means *no evidence either way*. Writing "TSFMs did not beat the baselines"
here would be a false statement.

## 4. The MCS view
Instead of "who beat the reference?", the MCS asks "who cannot be ruled out as best?".

In [ ]:
print(mcs.query("period == 'full' and ticker == 'POOLED' and target == 'rv'")
      [["horizon", "model", "mean_loss", "mcs_pvalue", "in_mcs"]].round(3).to_string(index=False))

## 5. A critique checklist
Ask these of *any* forecasting result, including ours:
1. **Data:** real or synthetic? Which period? Survivorship?
2. **Leakage:** could any input have come from after the origin?
3. **Sample size:** how many origins (T)? Any `small_sample` flags?
4. **Multiplicity:** how many tests were run, and was the claim corrected?
5. **Effect size:** is the CI narrow enough to matter, or just "significant"?
6. **Contamination:** could the model have seen the test period?
7. **Robustness:** does it hold with a rolling window? In the synthetic control?

Open `reports/dashboard/index.html` in a browser to explore the same tables interactively.

## ✅ Checkpoint
Write `verdict(mean_diff, p_holm, alpha=0.05)` that returns exactly the report's wording:
`"model better"`, `"model worse"`, or `"no detectable difference"`.

In [ ]:
def verdict(mean_diff, p_holm, alpha=0.05):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
from checker import check
check(verdict)

## Flashcards

Cover the answer, say it out loud, then check. `make flashcards` exports these to Anki.

1. **Q:** Where does every number in reports/RESULTS.md come from?  
   **A:** From stored tables in results/<run>/stats/*.parquet, listed with their SHA-256 in the Provenance section.
2. **Q:** What is the difference between an absent result and a null result?  
   **A:** Absent = the model could not be evaluated (UNAVAILABLE), so no evidence either way; null = evaluated, no detectable difference.
3. **Q:** A comparison has raw p = 0.01 but Holm p = 0.20. Is it a finding?  
   **A:** No. It does not survive the pre-specified multiple-testing correction.
4. **Q:** What does a relative loss of 0.95 [0.90, 1.01] mean?  
   **A:** The model's loss is 5% below the reference on average, but the 95% CI includes 1 (no difference).
5. **Q:** What does it mean if the MCS contains several models?  
   **A:** The data cannot distinguish them from the best model at the chosen confidence level.
6. **Q:** Why flag tests with fewer than 100 origins?  
   **A:** The DM test's actual size is above nominal in small samples, so marginal p-values are unreliable.
7. **Q:** Name four items of the critique checklist.  
   **A:** Real vs synthetic data, leakage, sample size, multiplicity (also effect size, contamination, robustness).

**Next:** open `lessons/10-writing-up/lesson.ipynb` (turning this into a short paper).
Tick lesson 09 in `lessons/PROGRESS.md`.